# Data Exploration

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

## References to use
- https://www.kaggle.com/code/juliencs/a-study-on-regression-applied-to-the-ames-dataset
- https://www.kaggle.com/code/dgawlik/house-prices-eda#Price-Segments

In [ ]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "data_raw"

EDA_DATA_DIR = DATA_DIR / "data_eda"
EDA_DATA_DIR.mkdir(exist_ok=True)

In [ ]:
# LOAD THE DATASET
df_raw = pd.read_csv(RAW_DATA_DIR / "train.csv")
df = df_raw.copy()

In [ ]:
# DATASET OVERVIEW
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
# CHECK NUMBER OF ROWS AND COLUMNS
df.shape

In [ ]:
# CHECK COLUMN NAMES (SPACES, CASING, WEIRD CHARACTERS?) 

df.columns = df.columns.str.upper()
columns_str = df.select_dtypes(include='string')

for col in columns_str.columns:
    df[col] = df[col].str.upper()

df.head()

The columns: `1STFLRSF`, `2NDFLRSF`, `3SSNPORCH` start with a number, can become a problem on the future

## 2. First look at the data

In [ ]:
# VIEW THE FIRST FEW ROWS
df.head()

In [ ]:
# VIEW A RANDOM SAMPLE OF ROWS 

df.sample(n=5) # You can define a random_state, if you prefer...

In [ ]:
# CHECK DTYPES OF EVERY COLUMN

df.info()

**FLAGGED COLUMNS:** None.

## 3. Quantify missing data


In [ ]:
# NULLS PER COLUMN AND PERCENTAGE OF NULLS PER COLUMN

nulls_per_column = df.isnull().sum()
percentage_nulls_per_column = df.isnull().mean()
nulls_per_column = pd.DataFrame(nulls_per_column)
nulls_per_column = nulls_per_column.rename(columns={0: 'SUM'})
nulls_per_column['%'] = percentage_nulls_per_column.round(4)*100
nulls_per_column = nulls_per_column.sort_values(by='SUM', ascending=False)

nulls_per_column[nulls_per_column['SUM'] > 0]


In [ ]:
# CHECK IF MISSINGNESS CLUSTERS

zero_null_values = nulls_per_column[nulls_per_column['SUM'] == 0].index
nullity_correlation = df.isnull().corr()
nullity_correlation = nullity_correlation.drop(index=zero_null_values, columns=zero_null_values)

plt.figure(figsize=(12,10))
ax = plt.axes()

sns.heatmap(nullity_correlation, annot=True, fmt='.1f', cmap='cividis', ax=ax)
ax.set_title('Nullity Correlation Heatmap')
plt.show()

**ANALYSIS**: The columns related to **garage**, like GARAGEQUAL, GARAGEFINISH, GARAGETYPE, GARAGEYRBLT, and GARAGECOND, have a strong relation of occurrence. The same phenomenon occurs in columns related to **basement**: BSMTFINTYPE2, BSMTEXPOSURE, BSMTCOND, BSMTQUAL, and BSMTFINTYPE1.

This phenomenon may indicate that an issue occurred in data collection. A fair conclusion is that these null values indicate that there are properties without a garage and basement that are classified with NULL VALUES.

In [ ]:
df.info()

In [ ]:
## CHECK FOR INVALID VALUES OF STRING COLUMNS

columns_str = df.select_dtypes(include='string')
for col in columns_str.columns:
   print(df[col].value_counts(), '\n')
   

**ANALYSIS**: `BLDGTYPE` / `TWNHS` is the only genuinely invalid entry in the 
dataset when compared against `data_description.txt`. Two remediation 
paths exist:

1. Rename `TWNHS` to `TWNHSI` (assumes all such rows are Inside units)
2. Merge `TWNHSE` and `TWNHS` into a single TOWNHOUSE category (avoids 
   the assumption, loses the End/Inside signal)

**OBSERVATION**: Confirmed an invalid value exists by comparing the data 
against `data_description.txt`.

## 4. Check for duplicates

In [ ]:
# CHECK FULLY DUPLICATED ROWS

duplicate_rows = df.duplicated()
duplicate_rows = pd.DataFrame(duplicate_rows)
duplicate_rows = duplicate_rows.rename(columns={0: 'DUPLICATE'})
duplicate_rows = duplicate_rows[duplicate_rows['DUPLICATE'] == True]

duplicate_rows

In [ ]:
# CHECK FULLY DUPLICATED ID ROWS

duplicate_id = df.duplicated(subset='ID')
duplicate_id = pd.DataFrame(duplicate_id)
duplicate_id = duplicate_id.rename(columns={0: 'DUPLICATE_ID'})
duplicate_id = duplicate_id[duplicate_id['DUPLICATE_ID'] == True]

duplicate_id

## 5. Look at each column's distribution

- Categorical columns: unique values and their counts
- Flag inconsistent labels (`"USA"` vs `"U.S.A"` vs `"usa"`)


In [ ]:
# NUMERIC COLUMNS: ANALYZE THE DISTRIBUTION, AND FLAG POSSIBLE ISSUES (INT)
columns_int = df.select_dtypes(include='int64')

for col in columns_int.columns:
    counts = pd.cut(df[col],bins=7, right=False, precision=0).value_counts()
    max_bin = (df[col].value_counts()).shape[0]
    if(max_bin >= 10 ): counts = pd.cut(df[col],bins=7, right=False, precision=0).value_counts()
    else: counts = df[col].value_counts()
    fig = px.bar(x = counts.index.sort_values().astype('str'), y=counts.values, title=col, width=800, height=400)
    fig.show()

**FLAG SPECIFIC COLUMNS**

- LOTAREA (concentrated)
- BSMTFINSF2 (presence of outliers)
- 1STFLRSF (presence of outliers)
- LOWQUALFINSF (concentrated + presence of outliers)
- GRLIVAREA (presence of outlier)
- BSMTHALFBATH (concentrated)
- KITCHENABVGR (concentrated)
- OPENPORCHSF (concentrated + presence of outliers)
- ENCLOSEDPORCH (concentrated + presence of outliers)
- 3SSNPORCH (concentrated + presence of outliers)
- SCREENPORCH (concentrated + presence of outliers)
- POOLAREA (concentrated + presence of outliers)
- MISCVAL (concentrated + presence of outliers)
- SALEPRICE (presence of outliers)

## 6. Check relationships between columns (lightly)
- Correlation between numeric columns (just to get a feel)
- Cross-tabs between categorical columns if relevant

## 7. Train/test consistency (if competition has both)
- Check that `train` and `test` have the same columns (except target)
- Check that dtypes match between `train` and `test` for shared columns
- Check that categorical columns have the same possible values in both — flag any category present in one but not the other
- Check that the null patterns are similar between `train` and `test` (a column missing only in test is a red flag)

## 8. Target variable exploration (train only)
- Look at the target's distribution (histogram for regression, value counts for classification)
- Check for class imbalance (classification) or skew (regression)
- Check for outliers or implausible values in the target itself
- Note whether the competition metric is sensitive to imbalance/skew (this affects strategy later)

## 9. Leakage red flags (just note, don't fix yet)
- For each column, ask: "would I actually know this value at prediction time?"
- Flag any column that looks like it was computed *after* the target/outcome happened
- Flag any column that correlates suspiciously well with the target (almost too good)
- Check the competition description/rules for explicitly forbidden features

## 10. Write down your findings
- Add a markdown cell summarizing what's wrong
- Add a markdown cell listing leakage suspects and train/test mismatches separately
- This becomes the to-do list for `02_data_cleaning.ipynb`